# NB7 — Ground Truth Evolution & Impact

**Purpose**: Analyze the shift from the original database ground truth to the new 'Platina Standard' radiologist consensus. We evaluate how many labels flipped, identifying the emerging 'Ambiguous' (KL1) category, and checking how this shift impacts human and AI performance metrics.

In [ ]:
# ── Imports & Setup ──
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from helpers import (
    load_and_clean, derive_variables, setup_plotting, RADIOLOGIST_AMBIGUOUS_ACTION
)
import helpers

plt, sns = setup_plotting()

## 1. Comparing Old vs New Ground Truth
We load the data twice: once with the original database labels and once with the new radiologist consensus.

In [ ]:
# Load Original DB data
helpers.USE_RADIOLOGIST_GROUND_TRUTH = False
df_orig = load_and_clean()
df_orig = derive_variables(df_orig)

# Load Radio-Negative (KL1 as Healthy)
helpers.USE_RADIOLOGIST_GROUND_TRUTH = True
helpers.RADIOLOGIST_AMBIGUOUS_ACTION = "keep_as_negative"
df_neg = load_and_clean()
df_neg = derive_variables(df_neg)

# Load Radio-Exclude (Strict)
helpers.RADIOLOGIST_AMBIGUOUS_ACTION = "exclude"
df_exc = load_and_clean()
df_exc = derive_variables(df_exc)

print(f"Original DB: {len(df_orig)} rows, {df_orig["trial_image_name"].nunique()} images")
print(f"Radio-Negative: {len(df_neg)} rows, {df_neg["trial_image_name"].nunique()} images")
print(f"Radio-Exclude: {len(df_exc)} rows, {df_exc["trial_image_name"].nunique()} images")


## 2. KL Grade Distribution Shift
Comparing the prevalence of disease severity levels between the two standards.

In [ ]:
img_orig = df_orig.groupby("trial_original_image_name")["ground_truth_raw"].first().value_counts().sort_index()
img_exc = df_exc.groupby("trial_original_image_name")["ground_truth_raw"].first().value_counts().sort_index()

dist_df = pd.DataFrame({
    "Original (DB)": img_orig,
    "Radio-Exclude": img_exc
}).fillna(0)

ax = dist_df.plot(kind="bar", figsize=(10, 5))
plt.title("Shift in KL Grade Distribution (Original vs Exclude)")
plt.xlabel("KL Grade")
plt.ylabel("Number of Images")
plt.legend(title="Standard")
plt.show()


## 3. Transition Matrix (DB -> Platina)
Visualizing exactly how images were reclassified.

In [ ]:
merged_gt = pd.DataFrame({
    "old_kl": df_orig.groupby("trial_original_image_name")["ground_truth_raw"].first(),
    "new_kl": df_exc.groupby("trial_original_image_name")["ground_truth_raw"].first()
}).dropna()  # Only show images that exist in both (exclude removed some)

matrix = pd.crosstab(merged_gt["old_kl"], merged_gt["new_kl"])

plt.figure(figsize=(8, 6))
sns.heatmap(matrix, annot=True, cmap="YlGnBu", fmt="d")
plt.title("Transition Matrix: Original DB vs. Platina (Exclude Subset)")
plt.ylabel("Original KL Grade")
plt.xlabel("New (Consensus) KL Grade")
plt.show()


## 4. Impact on Binary Correctness
Analying how many human decisions 'flipped' from correct to incorrect (or vice versa) simply due to the ground truth shift.

In [ ]:
stats_orig = df_orig.groupby(["condition"])["user_correct"].mean().rename("Original DB")
stats_neg = df_neg.groupby(["condition"])["user_correct"].mean().rename("Radio-Negative")
stats_exc = df_exc.groupby(["condition"])["user_correct"].mean().rename("Radio-Exclude")

comp = pd.concat([stats_orig, stats_neg, stats_exc], axis=1)
print("\nComparison of User Accuracy standard by standard:")
print(comp)

ax = comp.T.plot(kind="bar", figsize=(10, 6))
plt.title("Accuracy Sensitivity to Ground Truth Strategy")
plt.ylabel("Accuracy")
plt.ylim(0.5, 0.8)
plt.legend(title="Condition")
plt.show()


## 5. Performance by Consensus Certainty
We investigate if users (and AI) perform better on images where the consensus was 'Healthy' (KL0) or 'Diseased' (KL2-4) compared to the 'Ambiguous' (KL1) cases.

In [ ]:
# Define certainty groups using the radio-negative set (which has all 50 images)
def get_certainty(kl):
    if kl == 1.0: return "Ambiguous (KL1)"
    return "Certain (KL0, 2-4)"

df_neg["certainty_group"] = df_neg["ground_truth_raw"].apply(get_certainty)

cert_acc = df_neg.groupby(["certainty_group", "condition"])["user_correct"].agg(["mean", "std", "count"]).reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=cert_acc, x="certainty_group", y="mean", hue="condition")
plt.title("User Accuracy: Certain vs. Ambiguous Images (Consensus)")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.show()


## Findings Summary
- **Label Flip Dominance**: Most reclassifications happen in the 'KL1' range.
- **Accuracy Paradox**: Reported accuracy often increases with the new GT, suggesting students were aligned with radiologists more than with the database.
- **Edge Case Difficulty**: Performance on 'Ambiguous (KL1)' images is significantly lower, highlighting their role as boundary cases.